In [46]:

import cv2
import numpy as np
import time
import RPi.GPIO as GPIO

from collections import deque
from picamera import PiCamera
from picamera.array import PiRGBArray
from LOBOROBOT import LOBOROBOT

robot = LOBOROBOT()

# ========== GPIO ==========
TRIG = 20
ECHO = 21

GPIO.setwarnings(False)
GPIO.setmode(GPIO.BCM)
GPIO.setup(TRIG, GPIO.OUT, initial=GPIO.LOW)
GPIO.setup(ECHO, GPIO.IN)

# ========== 视觉参数 ==========
MIN_AREA = 100
TARGET_AREA = 365
AREA_DEAD_BAND = 50

CENTER_X = 160
DEAD_ZONE = 12

LOWER_BLUE = np.array([96, 157, 100])
UPPER_BLUE = np.array([120, 255, 255])

# ========== 电机参数 ==========
MAX_FORWARD = 30
MAX_STEER = 8
MAX_MOTOR_PWM = 30

# 物理电机最低有效 PWM，后续需要实测
MIN_MOTOR_PWM = 20

# 控制量每次最大变化
SPEED_STEP = 3

# 前进和转向比例系数
KP_DISTANCE = 0.12
KP_STEER = 0.10

CONTROL_PERIOD = 0.06
TEST_DURATION = 180

# 仅用于架空测试，不是地面安全停车距离
STOP_DISTANCE_CM = 15

# 已经通过你的测试确认
REVERSE_STEERING = True

# ========== 状态变量 ==========
camera = None
area_history = deque(maxlen=5)

current_a = 0.0
current_b = 0.0


def stop_all():
    for motor in range(4):
        robot.MotorStop(motor)


def safe_distance(timeout=0.03):
    GPIO.output(TRIG, GPIO.LOW)
    time.sleep(0.000002)

    GPIO.output(TRIG, GPIO.HIGH)
    time.sleep(0.00001)
    GPIO.output(TRIG, GPIO.LOW)

    deadline = time.monotonic() + timeout

    while GPIO.input(ECHO) == GPIO.LOW:
        if time.monotonic() >= deadline:
            return None

    start = time.monotonic()
    deadline = start + timeout

    while GPIO.input(ECHO) == GPIO.HIGH:
        if time.monotonic() >= deadline:
            return None

    distance = (
        time.monotonic() - start
    ) * 34300 / 2

    if 2 <= distance <= 400:
        return distance

    return None


def approach(current, target, step):
    if current < target:
        return min(current + step, target)

    return max(current - step, target)


def set_motor_signed(motor, speed):
    """
    将控制器输出映射为实际电机 PWM。

    speed = 0 时停止。
    非零时使用最低有效 PWM。
    """

    speed = float(
        np.clip(speed, -MAX_MOTOR_PWM, MAX_MOTOR_PWM)
    )

    if abs(speed) < 1:
        robot.MotorStop(motor)
        return

    # 保留速度差，同时补偿电机启动死区
    pwm = MIN_MOTOR_PWM + (
        MAX_MOTOR_PWM - MIN_MOTOR_PWM
    ) * abs(speed) / MAX_MOTOR_PWM

    pwm = int(
        np.clip(pwm, MIN_MOTOR_PWM, MAX_MOTOR_PWM)
    )

    if speed > 0:
        robot.MotorRun(motor, "forward", pwm)
    else:
        robot.MotorRun(motor, "backward", pwm)


def drive_smooth(a, b):
    """
    A 组：电机 0、2
    B 组：电机 1、3
    """

    set_motor_signed(0, a)
    set_motor_signed(2, a)

    set_motor_signed(1, b)
    set_motor_signed(3, b)


def calculate_forward(area):
    """
    距离比例控制：
    面积越小，目标通常越远。
    接近目标面积时停止前进。
    """

    area_error = TARGET_AREA - area

    if area_error <= AREA_DEAD_BAND:
        return 0.0

    forward = area_error * KP_DISTANCE

    return float(
        np.clip(forward, 0, MAX_FORWARD)
    )


def calculate_steer(error_x):
    """
    水平比例转向控制。
    """

    if abs(error_x) <= DEAD_ZONE:
        return 0.0

    steer = error_x * KP_STEER

    steer = float(
        np.clip(steer, -MAX_STEER, MAX_STEER)
    )

    if not REVERSE_STEERING:
        steer = -steer

    return steer


try:
    stop_all()

    # ========== 摄像头云台 ==========
    robot.set_servo_angle(10, 90)
    robot.set_servo_angle(9, 5)

    # ========== 摄像头 ==========
    camera = PiCamera()
    camera.resolution = (320, 240)
    camera.framerate = 15

    rawCapture = PiRGBArray(
        camera,
        size=(320, 240)
    )

    time.sleep(2)

    print("TASK 7.8 STARTED")
    print("Continuous visual tracking enabled")

    start_time = time.monotonic()
    last_control = 0

    for frame_data in camera.capture_continuous(
        rawCapture,
        format="bgr",
        use_video_port=True
    ):

        frame = frame_data.array.copy()
        rawCapture.truncate(0)

        now = time.monotonic()

        if now - start_time >= TEST_DURATION:
            break

        if now - last_control < CONTROL_PERIOD:
            continue

        last_control = now

        # ========== 蓝色目标检测 ==========
        hsv = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2HSV
        )

        mask = cv2.inRange(
            hsv,
            LOWER_BLUE,
            UPPER_BLUE
        )

        contours, _ = cv2.findContours(
            mask,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )

        found = False
        area = 0
        error_x = 0

        if contours:
            largest = max(
                contours,
                key=cv2.contourArea
            )

            area = cv2.contourArea(largest)

            if area >= MIN_AREA:
                found = True

                x, y, w, h = cv2.boundingRect(
                    largest
                )

                cx = x + w // 2
                error_x = cx - CENTER_X

                area_history.append(area)

        if not found:
            area_history.clear()

        stable_area = (
            float(np.median(area_history))
            if area_history
            else 0
        )

        # ========== 超声波 ==========
        distance_cm = safe_distance()

        # ========== 安全判断 ==========
        safe = (
            found
            and distance_cm is not None
            and distance_cm >= STOP_DISTANCE_CM
        )

        if not safe:
            current_a = 0.0
            current_b = 0.0

            stop_all()

            if not found:
                state = "TARGET_LOST"

            elif distance_cm is None:
                state = "SENSOR_ERROR"

            else:
                state = "OBSTACLE_STOP"

            forward = 0.0
            steer = 0.0

        else:
            # ========== 比例前进控制 ==========
            forward = calculate_forward(
                stable_area
            )

            # ========== 比例转向控制 ==========
            steer = calculate_steer(
                error_x
            )

            # ========== 差速混合 ==========
            target_a = forward - steer
            target_b = forward + steer

            target_a = float(
                np.clip(
                    target_a,
                    -MAX_MOTOR_PWM,
                    MAX_MOTOR_PWM
                )
            )

            target_b = float(
                np.clip(
                    target_b,
                    -MAX_MOTOR_PWM,
                    MAX_MOTOR_PWM
                )
            )

            # ========== 平滑速度控制 ==========
            current_a = approach(
                current_a,
                target_a,
                SPEED_STEP
            )

            current_b = approach(
                current_b,
                target_b,
                SPEED_STEP
            )

            # ========== 连续驱动 ==========
            drive_smooth(
                current_a,
                current_b
            )

            state = "TRACKING"

        print(
            f"{state} | "
            f"Area={stable_area:.0f} | "
            f"X={error_x} | "
            f"Distance={distance_cm} | "
            f"Forward={forward:.1f} | "
            f"Steer={steer:.1f} | "
            f"Motors=({current_a:.1f}, "
            f"{current_b:.1f})"
        )

except KeyboardInterrupt:
    print("Interrupted")

except Exception as e:
    print(
        "Error:",
        type(e).__name__,
        str(e)
    )

finally:
    stop_all()

    if camera is not None:
        camera.close()

    print("ALL MOTORS STOPPED")


TASK 7.8 STARTED
Continuous visual tracking enabled
TARGET_LOST | Area=0 | X=0 | Distance=None | Forward=0.0 | Steer=0.0 | Motors=(0.0, 0.0)
TARGET_LOST | Area=0 | X=0 | Distance=None | Forward=0.0 | Steer=0.0 | Motors=(0.0, 0.0)
TARGET_LOST | Area=0 | X=0 | Distance=None | Forward=0.0 | Steer=0.0 | Motors=(0.0, 0.0)
TARGET_LOST | Area=0 | X=0 | Distance=None | Forward=0.0 | Steer=0.0 | Motors=(0.0, 0.0)
TARGET_LOST | Area=0 | X=0 | Distance=175.43028263426095 | Forward=0.0 | Steer=0.0 | Motors=(0.0, 0.0)
TARGET_LOST | Area=0 | X=0 | Distance=None | Forward=0.0 | Steer=0.0 | Motors=(0.0, 0.0)
TARGET_LOST | Area=0 | X=0 | Distance=None | Forward=0.0 | Steer=0.0 | Motors=(0.0, 0.0)
TARGET_LOST | Area=0 | X=0 | Distance=None | Forward=0.0 | Steer=0.0 | Motors=(0.0, 0.0)
TARGET_LOST | Area=0 | X=0 | Distance=None | Forward=0.0 | Steer=0.0 | Motors=(0.0, 0.0)
TARGET_LOST | Area=0 | X=0 | Distance=149.01767054780066 | Forward=0.0 | Steer=0.0 | Motors=(0.0, 0.0)
TARGET_LOST | Area=0 | X=0 | D